# Migrant Archive — Colab Transcription

Transcribe a YouTube video using **whisperx large-v3** on Colab's free T4 GPU.

### What this notebook does
1. Clones the project repo (or uses uploaded files)
2. Downloads audio from YouTube via `yt-dlp`
3. Transcribes using `whisperx` (large-v3, GPU)
4. Saves the result as JSON to your Google Drive

### Estimated time
| Video length | Transcription time on T4 GPU |
|-------------|------------------------------|
| 5 min | ~15 seconds |
| 30 min | ~2 minutes |
| 2 hours | ~8-10 minutes |

### After this notebook
Download the JSON from Drive → place in `data/raw/whisper/` → run `rag_test.py --rebuild`

---
> For speaker labels, set a HuggingFace token in section 5.
---

## 1. Install Dependencies

In [ ]:
!pip install -q yt-dlp whisperx "transformers==4.48.3"
!apt-get update -qq && apt-get install -y -qq ffmpeg

# Downgrading transformers breaks numpy/scipy compatibility — reinstall
!pip install -q --force-reinstall numpy scipy
print("Dependencies ready.")

## 2. Mount Google Drive

This is where the JSON output will be saved. You'll need to authorize access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/migrant-archive"
LOCAL_AUDIO = "/content/audio"
os.makedirs(f"{DRIVE_ROOT}/output", exist_ok=True)
os.makedirs(LOCAL_AUDIO, exist_ok=True)

print(f"Drive mounted. Output -> {DRIVE_ROOT}/output/")
print(f"Local audio cache -> {LOCAL_AUDIO}/")

## 3. Get the Ingestion Code

Choose **one** method below. Option A (git clone) is recommended — it pulls all files with correct imports.

### Option A: Clone the repo

Use this if your repo is public. For private repos, use Option B.
If clone fails with a username error, switch to Option B.

In [ ]:
import sys
from pathlib import Path

# Clone the repo
REPO_URL = "https://github.com/Sebastianlopez-dev/migrant-archive.git"
!git clone {REPO_URL} /content/migrant-archive

# Add backend/core to Python path
sys.path.insert(0, str(Path("/content/migrant-archive/backend/core")))

print("Repo cloned and path configured")

### Option B: Upload files manually (private repos)

Upload these files from `backend/core/` and `data/`:
- `ingestion.py`
- `ingestion_audio.py`
- `ingestion_colab.py`
- `youtube-links.txt` (for batch — section 8)
- `cookies-yt.txt` (optional, only if iOS client fails)

Use the Files panel (left sidebar) to upload, then run this cell:

In [ ]:
# Run if using Option B (manual upload)
import sys
from pathlib import Path
sys.path.insert(0, "/content")
print("Manual upload path configured")

## 4. Configure Your Video

Set the YouTube URL and language below. For the FILMIG channel, use `es`.
For Catalan/Spanish code-switching, try `ca` or `es`.

In [ ]:
# ══════════════════════════════════════════
# EDIT THESE TWO VALUES
# ══════════════════════════════════════════

#VIDEO_URL = "https://www.youtube.com/watch?v=PASTE_VIDEO_ID_HERE"
VIDEO_URL = "https://youtu.be/CTmWjuQcvHY"
LANGUAGE  = "es"       # es, en, ca, or "" for auto-detect

# ══════════════════════════════════════════
# Advanced (usually leave as-is)
# ══════════════════════════════════════════

MODEL     = "large-v3"  # tiny, base, small, medium, large-v3
DEVICE    = "cuda"      # cuda (GPU) or cpu

print(f"Video: {VIDEO_URL}")
print(f"Language: {LANGUAGE}  |  Model: {MODEL}  |  Device: {DEVICE}")

## 5. HuggingFace Token (WhisperX)

WhisperX needs a HuggingFace token to download the speaker diarisation model
(pyannote/speaker-diarization-3.1).

1. Create a token at https://huggingface.co/settings/tokens (read-only is enough)
2. Accept terms at https://hf.co/pyannote/speaker-diarization-3.1
3. Run this cell to set it before transcription

In [ ]:
import os
#os.environ["HF_TOKEN"] = "hf_tu_token_aqui"
os.environ["HF_TOKEN"] = ""
# Verify it's set
if os.environ.get("HF_TOKEN", "").startswith("hf_"):
    print("HF_TOKEN ready for WhisperX diarisation")
else:
    print("... HF_TOKEN not set — transcription will work but speakers won't be labelled")

## 6. yt-dlp Setup (Player Client + Cookies)

Colab has no JavaScript runtime, so the default yt-dlp web client fails.
This cell patches yt-dlp to use the **iOS client** (no JS challenge).

If you have a `cookies-yt.txt` file uploaded, it will be used for auth.
Otherwise yt-dlp works without cookies for public videos.

In [ ]:
import ingestion
import yt_dlp
from pathlib import Path

COOKIE_FILE = "/content/cookies-yt.txt"
USE_COOKIES = Path(COOKIE_FILE).exists()

if USE_COOKIES:
    print(f"Cookies found: {COOKIE_FILE}")
else:
    print("No cookies file — using iOS client for public videos")

# ── Replace _fetch_metadata ──
def _fetch_metadata_fixed(video_url: str) -> dict:
    ydl_opts: dict = {
        "quiet": True,
        "skip_download": True,
        "extractor_args": {"youtube": {"player_client": ["ios"]}},
    }
    if USE_COOKIES:
        ydl_opts["cookiefile"] = COOKIE_FILE
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        return ydl.extract_info(video_url, download=False)

# ── Replace _download_audio ──
def _download_audio_fixed(video_url: str, output_dir: str = "data/audio") -> Path:
    audio_dir = Path(output_dir)
    audio_dir.mkdir(parents=True, exist_ok=True)

    info = _fetch_metadata_fixed(video_url)
    audio_path = audio_dir / f"{info['id']}.mp3"

    if audio_path.exists():
        if audio_path.stat().st_size > 1024:  # > 1KB means valid
            return audio_path
        audio_path.unlink()  # stale cache — delete and re-download

    ydl_opts: dict = {
        "format": "bestaudio/best",
        "outtmpl": str(audio_dir / "%(id)s.%(ext)s"),
        "noplaylist": True,
        "postprocessors": [{
            "key": "FFmpegExtractAudio",
            "preferredcodec": "mp3",
            "preferredquality": "192",
        }],
        "quiet": True,
        "no_warnings": True,
        "extractor_args": {"youtube": {"player_client": ["ios"]}},
    }
    if USE_COOKIES:
        ydl_opts["cookiefile"] = COOKIE_FILE

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.extract_info(video_url, download=True)

    return audio_path

# Monkeypatch the module
ingestion._fetch_metadata = _fetch_metadata_fixed
ingestion._download_audio = _download_audio_fixed

print("yt-dlp patched — iOS client active")

## 7. Run Transcription

This cell downloads the audio, runs Whisper, and saves the result.
For a 2-hour video, expect ~8-10 minutes on the free T4 GPU.

In [ ]:
import time
import json
from ingestion import VideoData, _fetch_metadata, _build_videodata, _download_audio
from ingestion_audio import _detect_device, _compute_type_for, _transcribe_audio

# ── Step 1: Fetch metadata ────────────────────────────────────────────
print("Fetching video metadata ...")
info = _fetch_metadata(VIDEO_URL)
video_id = info["id"]
title = info.get("title", "Unknown")
duration_mins = info.get("duration", 0) / 60

print(f"   Title: {title}")
print(f"   Duration: {duration_mins:.0f} min")
print(f"   Channel: {info.get('channel', 'Unknown')}")

# ── Step 2: Download audio ────────────────────────────────────────────
print(f"\nDownloading audio ...")
t0 = time.time()
audio_path = _download_audio(VIDEO_URL, output_dir=LOCAL_AUDIO)
print(f"   Done in {time.time() - t0:.0f}s -> {audio_path}")

# ── Step 3: Transcribe ────────────────────────────────────────────────
print(f"\nTranscribing with whisperx ({MODEL}, {DEVICE}) ...")
print(f"    This may take a few minutes for long videos ...")
t0 = time.time()

hf_token = os.environ.get("HF_TOKEN") or None
segments = _transcribe_audio(
    audio_path=str(audio_path),
    language=LANGUAGE,
    model_size=MODEL,
    device=DEVICE,
    hf_token=hf_token,
)

elapsed = time.time() - t0
rtf = elapsed / (info.get("duration", 1) / 60)  # real-time factor
print(f"   Done in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"   Speed: {rtf:.1f}x real-time")
print(f"   Segments: {len(segments)}")

# ── Step 4: Build VideoData & save ────────────────────────────────────
video_data = _build_videodata(info, segments)
output_path = video_data.save_json(output_dir=f"{DRIVE_ROOT}/output")

print(f"\nSaved to Drive: {output_path}")
print(f"   File size: {output_path.stat().st_size / 1024:.0f} KB")

# ── Step 5: Preview ───────────────────────────────────────────────────
print(f"\n{'─'*60}")
print(f"PREVIEW (first 300 chars):")
print(f"{'─'*60}")
print(video_data.full_text[:100])
print(f"{'─'*60}")

# ── Step 6: Summary ───────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"TRANSCRIPTION COMPLETE")
print(f"{'='*60}")
print(f"Video:    {title}")
print(f"Duration: {duration_mins:.0f} min")
print(f"Segments: {len(segments)}")
print(f"Chars:    {len(video_data.full_text):,}")
print(f"Speed:    {rtf:.1f}x real-time")
print(f"Saved:    {output_path}")
print(f"\n Next: Download this JSON → place in data/raw/whisper/ → rebuild index")

---

## 8. Batch Processing (Multiple Videos)

Upload `youtube-links.txt` to Colab (Files panel -> upload), then run this cell.
It reads all video URLs from the file and processes them one by one.

**Already-processed videos are skipped** -- add new URLs to the file and
re-run safely without re-transcribing everything.

> Use this for the first bulk run. Use **Section 7** for a single new video.

In [ ]:
import time
import os
from pathlib import Path
from ingestion import _fetch_metadata
from ingestion_audio import extract_single_video

# ══════════════════════════════════════════
# CONFIGURE
# ══════════════════════════════════════════
LINKS_FILE = "/content/youtube-links.txt"   # uploaded via Files panel
LANGUAGE   = "es"
MODEL      = "large-v3"
DEVICE     = "cuda"
OUTPUT_DIR = f"{DRIVE_ROOT}/output"
AUDIO_DIR  = LOCAL_AUDIO
SLEEP_SEC  = 45   # delay between videos to avoid rate-limit
hf_token   = os.environ.get("HF_TOKEN") or None

# ══════════════════════════════════════════
# Read links file (skip comments)
# ══════════════════════════════════════════
raw = Path(LINKS_FILE).read_text(encoding="utf-8").splitlines()
urls = [line.strip() for line in raw if line.strip() and not line.strip().startswith("#")]

print(f"{len(urls)} video(s) in links file")

# ══════════════════════════════════════════
# Process each video
# ══════════════════════════════════════════
ok, skipped, failed = 0, 0, 0

for i, url in enumerate(urls, 1):
    print(f"\n{'═'*60}")
    print(f"[{i}/{len(urls)}] {url}")
    print(f"{'═'*60}")

    # ── Check if already processed ──
    try:
        info = _fetch_metadata(url)
        video_id = info["id"]
    except Exception as e:
        print(f"   [ERROR] Metadata fetch failed: {e}")
        failed += 1
        continue

    json_path = Path(OUTPUT_DIR) / f"{video_id}.json"
    if json_path.exists():
        print(f"   [SKIP] Already exists: {json_path.name}")
        skipped += 1
        continue

    # ── Transcribe ──
    print(f"   Title: {info.get('title', 'Unknown')}")
    print(f"   Duration: {info.get('duration', 0) / 60:.0f} min")
    print(f"   Transcribing ...")

    t0 = time.time()
    try:
        data = extract_single_video(
            video_url=url,
            languages=[LANGUAGE],
            model_size=MODEL,
            device=DEVICE,
            output_dir=OUTPUT_DIR,
            audio_dir=AUDIO_DIR,
            hf_token=hf_token,
        )
        saved = data.save_json(output_dir=OUTPUT_DIR)
        elapsed = time.time() - t0
        print(f"   [OK] {len(data.transcript_segments)} segments in {elapsed:.0f}s")
        print(f"        Saved: {saved}")
        ok += 1
    except Exception as e:
        print(f"   [ERROR] Failed: {e}")
        failed += 1

    print(f"   Sleeping {SLEEP_SEC}s to avoid rate-limit ...")
    time.sleep(SLEEP_SEC)

# ══════════════════════════════════════════
# Summary
# ══════════════════════════════════════════
print(f"\n{'='*60}")
print(f"BATCH COMPLETE")
print(f"{'='*60}")
print(f"  [OK]    Processed: {ok}")
print(f"  [SKIP]  Skipped:   {skipped}")
print(f"  [ERROR] Failed:    {failed}")
print(f"\n  Output: {OUTPUT_DIR}/")
print(f"\n  Next: Download JSONs from Drive -> place in data/raw/whisper/ -> rebuild index")


## 9. Download the Result

The JSON is saved in your Google Drive at `migrant-archive/output/{video_id}.json`.

**Option A**: Download directly from [drive.google.com](https://drive.google.com) → My Drive → migrant-archive → output

**Option B**: Run this cell to download via Colab:

In [ ]:
from google.colab import files

# Download the JSON file to your local machine
files.download(str(output_path))

print("\nPlace this file in: migrant-archive/data/raw/whisper/")
print("Then run: python backend/scripts/rag_test.py --rebuild")

---

## 10. Troubleshooting

| Problem | Fix |
|---------|-----|
| `n challenge solving failed` | Re-run section 6 (patches yt-dlp to iOS client) |
| `transformers` / `Pipeline` import error | Re-run section 1 (pins transformers==4.48.3) |
| `yt-dlp` says video unavailable | Video may be private/age-restricted. Try another URL. |
| `No such file or directory` on mp3 | Stale cache from a failed run. Delete `/content/audio/*` and re-run. |
| Out of memory (OOM) | Switch to `MODEL = "medium"` — still excellent quality, less RAM. |
| GPU not available | Colab T4 quota exhausted. Wait a few hours or use `DEVICE = "cpu"` (much slower). |
| `whisperx` import error | Re-run the install cell. Colab sometimes loses packages. |
| Drive mount fails | Re-run the mount cell. Make sure you're logged into the correct Google account. |